# Milestone - Unit 2: the popularity path joins the recommender, and we measure its bias

## 1. Load the catalog and interactions

The same generated substrate Unit 1 built the scoreboard on. `bookrec.generated_dir()` reports a
clear error if `recsys/data/generated/` has not been generated yet.

In [ ]:
import json
from collections import defaultdict

import bookrec
import pandas as pd
from bookrec.protocol import BaseRetrievalPath, Candidate

K = 10
RANDOM_SEED = 0

data_dir = bookrec.generated_dir()
catalog = bookrec.load_catalog(data_dir / "catalog.csv.gz")
catalog_ids = sorted(catalog)

cold = json.loads((data_dir / "cold_partitions.json").read_text())
cold_readers = cold["cold_readers"]

interactions_path = data_dir / "interactions.csv.gz"
rows = pd.read_csv(interactions_path).to_dict("records")
len(catalog_ids), len(rows)

## 2. The popularity path vs the random floor

Build, fit, and register the `PopularityRetrievalPath` -- the first learned path in the system. It
counts only positive `train` interactions per item (val/test rows and `label == 0` rows are leakage
and never touch the counts), then recommends the most-counted unseen books to everyone. We score it
on Unit 1's frozen `val` scoreboard against the seeded random floor, cold readers excluded.

In [ ]:
popularity = bookrec.PopularityRetrievalPath().fit(rows, catalog=catalog_ids)

registry = bookrec.PathRegistry()
registry.register(popularity)

random_floor = bookrec.RandomRetrievalPath(catalog_ids, seed=RANDOM_SEED)

popularity_board = bookrec.run_validation_scoreboard(
    popularity, interactions_path, catalog_ids=catalog_ids, k=K, cold_readers=cold_readers
)
random_board = bookrec.run_validation_scoreboard(
    random_floor, interactions_path, catalog_ids=catalog_ids, k=K, cold_readers=cold_readers
)

{
    "registered": registry.get("popularity").artifact_name(),
    "items_counted": len(popularity.counts),
    "scored_readers": popularity_board.readers,
    "popularity_hit_at_10": round(popularity_board.hit_rate_at_k, 4),
    "random_floor_hit_at_10": round(random_board.hit_rate_at_k, 4),
}

The count path clears the floor by roughly an order of magnitude (~0.11 vs ~0.012): recommending
what most readers already liked is a real, honest baseline, and it earns its place.

## 3. The weighted-rating quality view scores far below popularity on `val`

Popularity ranks by raw count. The *quality* question is different: of the readers who were exposed
to a book, what fraction liked it? `weighted_rating` is the Bayesian-shrinkage estimate
`(v*R + m*C)/(v + m)` -- an item's own positive rate `R` over its exposure `v`, pulled toward the
global rate `C` by `m` pseudo-counts so thin evidence is not mistaken for quality.

It surfaces different books than popularity does. But scored on the SAME `val` holdout it lands far
below popularity -- because exposure in the holdout is popularity-weighted (taste-aware, but with a
popularity factor riding along), so the held-out positives lean toward the already-popular. A
quality ranking that ignores both popularity and the reader is only barely correlated with those
held-out positives. That is popularity bias living inside the metric.

In [ ]:
train_positives = defaultdict(int)
train_exposures = defaultdict(int)
for row in rows:
    if row["split"] == "train":
        train_exposures[row["item_id"]] += 1
        if row["label"] == 1:
            train_positives[row["item_id"]] += 1

positives = [train_positives.get(item_id, 0) for item_id in catalog_ids]
exposures = [train_exposures.get(item_id, 0) for item_id in catalog_ids]
quality_scores = dict(zip(catalog_ids, bookrec.weighted_rating(positives, exposures, m=10.0)))


class WeightedRatingView(BaseRetrievalPath):
    """Rank the catalog by the weighted-rating quality score; reader-independent, so it can be
    scored on the same frozen val scoreboard as the popularity path."""

    def __init__(self, scores):
        super().__init__(name="weighted-rating", version="1", _fitted=True)
        self._scores = dict(scores)

    def retrieve(self, query, context, k):
        seen = set(context.get("seen", ())) if isinstance(context, dict) else set()
        candidates = [
            Candidate(item_id, float(score), self.name)
            for item_id, score in self._scores.items()
            if item_id not in seen
        ]
        return self._finish(candidates, k)


quality_view = WeightedRatingView(quality_scores)
quality_board = bookrec.run_validation_scoreboard(
    quality_view, interactions_path, catalog_ids=catalog_ids, k=K, cold_readers=cold_readers
)

top_popularity = [item for item, _ in sorted(popularity.counts.items(), key=lambda kv: (-kv[1], kv[0]))[:5]]
top_quality = [item for item, _ in sorted(quality_scores.items(), key=lambda kv: (-kv[1], kv[0]))[:5]]

{
    "top_5_by_popularity": top_popularity,
    "top_5_by_quality": top_quality,
    "quality_view_hit_at_10": round(quality_board.hit_rate_at_k, 4),
    "popularity_hit_at_10": round(popularity_board.hit_rate_at_k, 4),
}

## 4. Popularity bias: catalog coverage and head-share

A scoreboard winner can still be a bad citizen of the catalog. The head set is the top 10% of items
by train positive count; `catalog_coverage` is the fraction of the catalog that ever reaches a
reader, and `head_share` is the fraction of recommendation slots filled by head items. We measure
both for the popularity path and the random floor over the SAME scored readers.

In [ ]:
head_ids = bookrec.head_ids_from_counts(popularity.counts, catalog_ids, fraction=0.10)


def collect_top_k(path):
    """Reproduce the scoreboard's scored-reader set and keep each reader's top-k list."""
    catalog_set = set(catalog_ids)
    cold_set = set(cold_readers)
    seen_by = defaultdict(set)
    val_by = defaultdict(set)
    for row in rows:
        if row["item_id"] not in catalog_set or row["label"] != 1:
            continue
        if row["split"] == "train":
            seen_by[row["reader_id"]].add(row["item_id"])
        elif row["split"] == "val":
            val_by[row["reader_id"]].add(row["item_id"])
    recommendations = []
    for reader_id in sorted(val_by):
        seen = seen_by[reader_id]
        if reader_id in cold_set or not (val_by[reader_id] - seen):
            continue
        candidates = path.retrieve(reader_id, {"seen": seen}, K)
        recommendations.append(bookrec.rank(candidates, n=K, exclude=seen))
    return recommendations


popularity_recs = collect_top_k(popularity)
random_recs = collect_top_k(random_floor)

{
    "head_set_size": len(head_ids),
    "popularity_coverage": round(bookrec.catalog_coverage(popularity_recs, catalog_ids), 4),
    "random_coverage": round(bookrec.catalog_coverage(random_recs, catalog_ids), 4),
    "popularity_head_share": round(bookrec.head_share(popularity_recs, head_ids), 4),
    "random_head_share": round(bookrec.head_share(random_recs, head_ids), 4),
}

The popularity "winner" serves almost none of the catalog (coverage ~0.008) and every slot it fills
is a head item (head-share 1.0), while the random floor spreads across ~0.92 of the catalog with a
head-share near the 10% head fraction. The system now recommends better than chance -- and we can
see exactly how narrow that win is. Personalisation (Unit 4) and beyond-accuracy metrics (Unit 13)
are the answer to this bias.